# Retrival Notebook 
  
Dieses Notebook dient dazu die Embeddings aus 04_embeddings.ipynb zu vergleichen.

In [ ]:
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm


def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")


def find_upwards(name):
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


CFG_FILE = find_upwards("config.yaml")
assert CFG_FILE, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(CFG_FILE.read_text())

METHOD = CFG["vpr"]["method"]
MODEL_ID = CFG["vpr"]["models"][METHOD]
RETRIEVAL_METHOD = CFG["retrieval"]["method"]
ADAPTER = CFG["vpr"].get("adapter", "none")

PROJECT_ROOT = find_project_root()
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings" / METHOD
RESULT_DIR = PROJECT_ROOT / "results"
RETRIEVAL_DIR = RESULT_DIR / "retrieval" / METHOD
RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)

if ADAPTER == "none" or ADAPTER == "None":
    EMBEDDING_NAME = METHOD
else:
    EMBEDDING_NAME = f"{METHOD}_{ADAPTER}"

embedding_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_embeddings.npy"
metadata_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_metadata.parquet"

embeddings = np.load(embedding_path)
embedding_metadata = pd.read_parquet(metadata_path)


# Weiter Infos
# print(embedding_metadata.head())
print("Emebddings")
print(f"Shape:      {embeddings.shape}")
print(f"DataType:   {embeddings.dtype}")
print()
print("Metadata")
print(f"Shape:      {embedding_metadata.shape}")

# Trennen und Assertions

In [ ]:
database_mask = (embedding_metadata["split"] == "database").to_numpy()
query_mask = (embedding_metadata["split"] == "query").to_numpy()

database_embeddings = embeddings[database_mask]
query_embeddings = embeddings[query_mask]

database_metadata = embedding_metadata[database_mask].reset_index(drop=True)
query_metadata = embedding_metadata[query_mask].reset_index(drop=True)


assert len(embeddings) == len(embedding_metadata)
assert len(database_embeddings) == len(database_metadata)
assert len(query_embeddings) == len(query_metadata)


print(f"Database Shape:     {database_embeddings.shape}")
print(f"Query Shape:        {query_embeddings.shape}")

# Retrieval Funktionen

In [ ]:
# Use Nump to retrieve
def retrieve_numpy(query_embedding ,database_embeddings, top_k = 10):

    similarities = database_embeddings @ query_embedding
    ranking = np.argsort(similarities)[::-1]

    top_indices = ranking[:top_k]
    top_similarities = similarities[top_indices]

    return top_indices, top_similarities



# Create FAISS Databse
def create_faiss(database_embeddings):

    dim = database_embeddings.shape[1]

    index = faiss.IndexFlatIP(dim)
    index.add(database_embeddings.astype(np.float32))

    return index



# Use FAISS to retrieve
def retrieve_faiss(query_embedding, index, top_k = 10):

    scores, indices = index.search(query_embedding.reshape(1, -1).astype(np.float32), top_k)

    return indices[0], scores[0]



# Retrieve the method in config fiel
def retrieve(query_embedding, database_embeddings, top_k=10, method="numpy", faiss_index=None):

    if method == "numpy":
        return retrieve_numpy(query_embedding, database_embeddings, top_k)

    if method == "faiss":
        return retrieve_faiss(query_embedding, faiss_index, top_k)

    raise ValueError(f"Nicht angegebene oder implementierte Retrivale Methode: {method}")


In [ ]:
# für nur eine Testquery
# query_embedding = query_embeddings[0]

if RETRIEVAL_METHOD == "faiss":
    faiss_index = create_faiss(database_embeddings)
else:
    faiss_index = None

TOP_K = 50
retrieval_results = []

for query_index, query_embedding in tqdm(enumerate(query_embeddings), total = len(query_embeddings), desc= "Retrieval"):

    indices, similarities = retrieve(
        query_embedding,
        database_embeddings,
        top_k = TOP_K,
        method = RETRIEVAL_METHOD,
        faiss_index = faiss_index,
    )

    retrieval_results.append({
        "query_index": query_index,
        "indices": indices,
        "similarities": similarities
    })



In [ ]:
query_index = 0

indices = retrieval_results[query_index]["indices"]
similarities = retrieval_results[query_index]["similarities"]

query_results = database_metadata.iloc[indices].copy()
query_results["similarity"] = similarities


# Auskommentieren für mehr Infos
# print(query_metadata.columns.tolist())
# print(query_metadata.head().to_string())
# print(database_metadata.head().to_string())
print(query_results)

In [ ]:
np.savez(
    RETRIEVAL_DIR / f"{EMBEDDING_NAME}_retrieval.npz",
    indices=np.array([r["indices"] for r in retrieval_results]),
    similarities=np.array([r["similarities"] for r in retrieval_results]),
)

print(f"Saved to {RETRIEVAL_DIR}")
